In [ ]:
# Ligar o autoreload
%load_ext autoreload
%autoreload 2

In [1]:
REPO_URL = "https://github.com/anacaetano02/Deep-Learning-Visao-Computacional.git"
SUBPASTA = "A1_vision_transformers"
REPO_DIR = "/content/repo"

if not REPO_URL:
    raise ValueError("Defina REPO_URL com a URL do seu repositório git antes de continuar.")

import os
import subprocess

PROJECT_DIR = os.path.join(REPO_DIR, SUBPASTA)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # --filter=blob:none: baixa só os metadados de commit/árvore, os
    # arquivos (blobs) de fora do sparse-checkout nunca são transferidos.
    # --sparse liga o modo cone do sparse-checkout (mais rápido e mais
    # simples de configurar que listar padrões manualmente).
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--sparse", "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )
    subprocess.run(["git", "sparse-checkout", "set", SUBPASTA], cwd=REPO_DIR, check=True)
else:
    # Repositório já clonado nesta sessão do Colab (variável de estado que
    # sobrevive a reexecuções de célula, só não a reinício de runtime) —
    # git pull traz commits novos em vez de simplesmente pular, para
    # mudanças feitas no código depois do clone inicial não ficarem presas
    # do lado de fora.
    print(f"Repositório já clonado em '{REPO_DIR}' — buscando mudanças mais recentes...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)


In [ ]:
# Célula de atualização, reexecutada a cada push
# !git -C "$REPO_DIR" pull --ff-only
# !git -C "$REPO_DIR" log -1 --oneline

In [2]:
# Onde salvar dados e artefatos desta execução.
SALVAR_NO_DRIVE = False  # AJUSTE conforme sua preferência.

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # AJUSTE: caminho da pasta de saída dentro do seu Drive.
    BASE_DIR = "/content/drive/MyDrive/deep_learning_visao_computacional/A1_vision_transformers"
else:
    BASE_DIR = "/content/A1_vision_transformers_saida"

BASE_DIR = os.path.abspath(BASE_DIR)
print(f"BASE_DIR = {BASE_DIR} (Drive: {SALVAR_NO_DRIVE})")


BASE_DIR = /content/A1_vision_transformers_saida (Drive: False)


In [3]:
# Estrutura de saída (Drive/BASE_DIR) + cópia local das imagens (sempre /content).
DIR_DADOS_RAW = os.path.join(BASE_DIR, "data", "raw")
DIR_DADOS_PROCESSADOS = os.path.join(BASE_DIR, "data", "processed")
DIR_OUTPUTS = os.path.join(BASE_DIR, "outputs")
DIR_CHECKPOINTS = os.path.join(DIR_OUTPUTS, "checkpoints")
DIR_REPORT_ASSETS = os.path.join(DIR_OUTPUTS, "report_assets")
DIR_RUNS = os.path.join(BASE_DIR, "runs")  # logs do TensorBoard

LOCAL_DIR = "/content/local_data"  # nunca no Drive — ver seção 2 acima

for d in (DIR_DADOS_RAW, DIR_DADOS_PROCESSADOS, DIR_CHECKPOINTS, DIR_REPORT_ASSETS, DIR_RUNS):
    os.makedirs(d, exist_ok=True)

print("Diretórios prontos:")
for d in (DIR_DADOS_RAW, DIR_DADOS_PROCESSADOS, DIR_CHECKPOINTS, DIR_REPORT_ASSETS, DIR_RUNS, LOCAL_DIR):
    print(f"  {d}")


Diretórios prontos:
  /content/A1_vision_transformers_saida/data/raw
  /content/A1_vision_transformers_saida/data/processed
  /content/A1_vision_transformers_saida/outputs/checkpoints
  /content/A1_vision_transformers_saida/outputs/report_assets
  /content/A1_vision_transformers_saida/runs
  /content/local_data


In [5]:
import sys
import numpy as np
import polars as pl
import torch


if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from src.data import analyze_skin_cancer_data



print("Módulos do projeto importados com sucesso.")

Módulos do projeto importados com sucesso.


In [8]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(torch.cuda.get_device_name(0))

from datetime import datetime
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID = {RUN_ID}")


Tesla T4
RUN_ID = 20260926_202324
